This tutorial can be followed through your browser, using Colab [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/main/tutorials/2.1_Train_SB3_Policy.ipynb).

# 2.1 — Train a Policy with Stable-Baselines3 (SB3)

This notebook trains a PPO policy on the elbow pose task using Stable-Baselines3, the most common modern RL library for MuJoCo environments.

**What you'll learn:**
- How to wrap a MyoSuite environment for SB3
- How to train, save, and reload a policy
- How to evaluate a trained policy with deterministic actions

**Prerequisites:** Completed notebook 1.1 · `pip install stable-baselines3`

> **Note:** `total_timesteps=1000` is intentionally short for a quick smoke-test — the policy will not converge. For a policy that actually solves the task, use at least `total_timesteps=500_000`. Training time on CPU: ~5 min for 500k steps.

In [ ]:
# Optional: If running in Colab, uncomment below to make
#           MyoSuite available in your runtime.
full_training = False
# !pip install git+https://github.com/MyoHub/myosuite
# !pip install stable-baselines3[extra]
# !apt install ffmpeg
# import os
# os.environ['MUJOCO_GL']='egl'
# full_training = True

In [ ]:
from myosuite.utils import gym
import numpy as np
import os
from myosuite.utils.video_io import show_video
from myosuite.utils.video_io import write_video


In [ ]:
env = gym.make('myoElbowPose1D6MRandom-v0', render_mode='rgb_array')

env.reset()

In [ ]:
from stable_baselines3 import PPO

model = PPO("MlpPolicy", env, verbose=0 if not full_training else 2,
            tensorboard_log="./myoElbowPose1D6MRandom-v0/" if full_training else None)

print("========================================")
print("Starting policy learning")
print("========================================")

model.learn(total_timesteps=1000 if not full_training else 200_000)  # smoke-test only; use ≥500_000 for full convergence

print("========================================")
print("Job Finished.")
print("========================================")

model.save('files/2.1/ElbowPose_policy')


In [ ]:
policy = "files/2.1/ElbowPose_policy.zip"

pi = PPO.load(policy)

# define a discrete sequence of positions to test
AngleSequence = [60, 30, 30, 60, 80, 80, 60, 30, 80, 30, 80, 60]
env.reset()
frames = []
for ep in range(len(AngleSequence)):
    print("Ep {} of {} testing angle {}".format(ep, len(AngleSequence), AngleSequence[ep]))
    env.unwrapped.target_jnt_value = [np.deg2rad(AngleSequence[int(ep)])]
    env.unwrapped.target_type = 'fixed'
    env.unwrapped.weight_range=(0,0)
    env.unwrapped.update_target()
    for _ in range(40):
        frames.append(env.render())
        o = env.unwrapped.get_obs()
        a = pi.predict(o)[0]
        next_o, r, done, *_, ifo = env.step(a) # take an action based on the current observation
env.close()

os.makedirs('videos', exist_ok=True)
# make a local copy
write_video('videos/arm.mp4', np.asarray(frames), outputdict={"-pix_fmt": "yuv420p"})
show_video('videos/arm.mp4')

In [ ]:
# Optional: Visualise learning curve and metrics using tensorboard
if full_training:
    %load_ext tensorboard
    %tensorboard --logdir ./myoElbowPose1D6MRandom-v0/

In [ ]:
print('Done. Policy saved under files/2.1/ElbowPose_policy.zip')